In [12]:
import os
os.getcwd()

'C:\\Users\\Mateusz\\Documents\\ML_testing\\Neural-Classification-of-Erythrocyte-Anomalies\\optuna'

In [4]:
# --- Cell: Inspect Database Content ---
import sqlite3
import optuna
import pandas as pd
import os
import matplotlib.pyplot as plt
import plotly


#---------Configuration---------------------
db_folder = "." # Upewnij się, że folder jest poprawny
#db_file = "Malaria_Stage2_Finetune.db" # Nazwa Twojej bazy
db_file = "Malaria_Stage1_Frozen.db"
db_path = os.path.join(db_folder, db_file)

# Optuna needs an absolute path for SQLite to avoid errors
storage_url = f"sqlite:///{os.path.abspath(db_path)}"

if not os.path.exists(db_path):
    print(f"❌ File {db_path} not exist. Check path.")
else:
    print(f"✅ Connected with database: {db_file}\n")

    # --- Part 1: List of available studies ---
    print("--- 1. List of studies in database ---")
    try:
        # Download summary of all studies in the .db file
        summaries = optuna.get_all_study_summaries(storage=storage_url)

        if not summaries:
            print("Database is empty (no experiments).")
        else:
            studies_df = []
            for s in summaries:
                studies_df.append({
                    "Name (ID)": s.study_name,
                    "Trials": s.n_trials,
                    "Start": s.datetime_start.strftime("%Y-%m-%d %H:%M"),
                    "Direction": str(s.direction)
                })

            display(pd.DataFrame(studies_df))

            # --- Part 2: Analiza konkretnego badania ---
            # Automatycznie wybieramy pierwsze badanie z listy (lub wpisz nazwę ręcznie)
            study_name = summaries[0].study_name
            # study_name = "Malaria_Stage2_Finetune" # <--- Odkomentuj i wpisz, jeśli chcesz konkretne

            print(f"\n--- 2. Szczegóły badania: '{study_name}' ---")

            # Ładujemy badanie
            study = optuna.load_study(study_name=study_name, storage=storage_url)

            # A. Najlepszy wynik
            try:
                print(f"Najlepszy wynik (Val Acc): {study.best_value:.4f}")
                print(" Najlepsze parametry:")
                for k, v in study.best_params.items():
                    print(f"     - {k}: {v}")
            except ValueError:
                print("   Brak zakończonych prób zakończonych sukcesem.")

            # B. Tabela wyników (Pandas)
            # To jest kluczowe! trials_dataframe() wyciąga wszystko do tabeli
            df = study.trials_dataframe()

            # Filtrujemy i czyścimy tabelę dla czytelności
            # Sortujemy od najlepszego wyniku (dla maximize)
            if 'value' in df.columns:
                df = df.sort_values(by='value', ascending=False)

            # Wyświetlamy tylko ciekawe kolumny (parametry i wyniki)
            # Kolumny zaczynające się od 'params_' to hiperparametry
            # Kolumny zaczynające się od 'user_attrs_' to Twoje dodatkowe metryki (train_acc itp.)
            cols_to_show = ['number', 'value', 'state'] + \
                           [c for c in df.columns if c.startswith('params_')] + \
                           [c for c in df.columns if c.startswith('user_attrs_')]

            print(f"\n--- 3. Tabela wyników (Top 10) ---")
            display(df[cols_to_show].head(10))

            # --- CZĘŚĆ 3: Szybkie Wykresy ---
            print("\n--- 4. Wizualizacja Historii ---")
            # Historia optymalizacji (czy wynik rósł w czasie?)
            try:
                fig1 = optuna.visualization.plot_optimization_history(study)
                fig1.show()

                # Ważność parametrów (co miało największy wpływ?)
                if len(study.trials) > 2:
                    fig2 = optuna.visualization.plot_param_importances(study)
                    fig2.show()
            except Exception as e:
                print(f"Nie można wygenerować wykresów (za mało danych lub brak plotly): {e}")

    except Exception as e:
        print(f"Wystąpił błąd podczas odczytu Optuny: {e}")

✅ Connected with database: Malaria_Stage1_Frozen.db

--- 1. List of studies in database ---


,Name (ID),Trials,Start,Direction
0,Malaria_Stage1_Frozen,68,2025-12-06 16:39,2



--- 2. Szczegóły badania: 'Malaria_Stage1_Frozen' ---
Najlepszy wynik (Val Acc): 0.9546
 Najlepsze parametry:
     - base_model: resnet50
     - n_layers: 4
     - hidden_dim: 128
     - dropout: 0.5
     - apply_dropout: False
     - batch_size: 32
     - learning_rate: 0.003665044026015035
     - optimizer: Adam

--- 3. Tabela wyników (Top 10) ---


,number,value,state,params_apply_dropout,params_base_model,params_batch_size,params_dropout,params_hidden_dim,params_learning_rate,params_n_layers,params_optimizer,user_attrs_overfit_gap,user_attrs_train_acc,user_attrs_train_loss
24,24,0.954628,COMPLETE,False,resnet50,32,0.5,128,0.003665,4,Adam,-0.017284,0.937344,0.169260
26,26,0.954174,COMPLETE,True,resnet50,16,0.3,64,0.001122,3,Adam,-0.023067,0.931107,0.189251
67,67,0.950998,COMPLETE,False,resnet50,16,0.5,128,0.001510,4,Adam,-0.015979,0.935019,0.177335
12,12,0.950544,COMPLETE,True,resnet50,16,0.2,64,0.003399,3,Adam,-0.024201,0.926344,0.207049
54,54,0.949637,COMPLETE,True,resnet50,16,0.5,256,0.001547,4,Adamax,-0.017623,0.932014,0.193124
11,11,0.949183,COMPLETE,True,resnet50,64,0.2,64,0.009388,2,Adagrad,-0.009798,0.939385,0.167428
51,51,0.949183,COMPLETE,True,resnet50,16,0.5,256,0.000903,4,Adamax,-0.017680,0.931504,0.189857
7,7,0.947822,COMPLETE,False,resnet50,32,0.5,128,0.004178,4,Adam,-0.011215,0.936607,0.169279
21,21,0.946915,COMPLETE,True,vgg16,16,0.2,256,0.000213,2,RMSprop,-0.009571,0.937344,0.175096
6,6,0.946461,COMPLETE,True,resnet50,64,0.1,64,0.007161,2,Adagrad,-0.006962,0.939499,0.162912



--- 4. Wizualizacja Historii ---
